# Solución del caso final de asignatura

## 1) Qué primitivas son vulnerables

- RSA: vulnerable frente a Shor.
- ECDSA y ECDH: también vulnerables frente a Shor.
- AES-256 y SHA-256 siguen siendo útiles, aunque sufren una pérdida de fuerza contra Grover. En la práctica se recomienda usar tamaños mayores como AES-256 y SHA-384/512.
- Los certificados X.509 con algoritmos clásicos también deberían ser revisados y evolucionados para soportar certificados híbridos.

## 2) Propuesta de diseño híbrido

Un diseño razonable sería:

- KEM post-cuántico: ML-KEM o HQC según la estrategia de adopción y compatibilidad.
- Cifrado simétrico autenticado: AES-256-GCM o ChaCha20-Poly1305.
- KDF: HKDF para derivar una clave de sesión a partir del secreto compartido.
- Firma: ML-DSA o SLH-DSA según requisitos de tamaño y latencia.
- Certificados híbridos: una parte clásica + otra post-cuántica, para compatibilidad gradual.

Esto permite mantener una seguridad robusta y una transición gradual sin romper el sistema operativo.

## 3) Qué firma elegir

Para un sistema académico con firma de documentos y validación general, la opción más natural es `ML-DSA` porque ofrece un equilibrio sólido entre seguridad, implementación y soporte estándar.

`SLH-DSA` puede ser útil en entornos con necesidad de diversidad matemática o requisitos especiales, pero suele tener firmas más grandes.

## 4) Servicios prioritarios

Los servicios más críticos son los de autenticación, la firma de documentos, las plataformas de matrícula y los accesos federados. También merece prioridad la infraestructura de certificados, porque si la validación de identidad se debilita, el resto del sistema queda expuesto. Los servicios menos críticos pueden migrarse en una fase posterior.

## 5) Plan de migración

1. Inventariar algoritmos, certificados y dependencias de los servicios.
2. Crear un entorno de prueba con protocolos híbridos.
3. Validar autenticación, firma y cifrado con un conjunto realista de escenarios.
4. Desplegar la pila híbrida en producción con monitorización y fallback.
5. Retirar progresivamente los algoritmos clásicos cuando la infraestructura lo permita.

## 6) Prototipo mínimamente ampliado en Python

In [ ]:
import hashlib

classical_secret = 'clave_clasica'
pqc_secret = 'clave_pqc'

def derive_key(k1, k2):
    seed = (k1 + ':' + k2).encode('utf-8')
    return hashlib.sha256(seed).hexdigest()

def encrypt_and_decrypt(message, key):
    nonce = 'nonce-demo'
    ciphertext = (message + ':' + nonce + ':' + key)[:len(message)]
    return ciphertext

final_key = derive_key(classical_secret, pqc_secret)
message = 'documento_escuela'
cipher = encrypt_and_decrypt(message, final_key)
print('Clave híbrida:', final_key)
print('Ciphertext simulado:', cipher)

# La idea del prototipo es demostrar la derivación de una clave híbrida y la estructura del protocolo.

## 5) Plan de migración

1. Inventariar algoritmos, certificados, claves y vida útil de los datos.
2. Probar un protocolo híbrido con KEM post-cuántico y AES-GCM.
3. Migrar la firma de documentos a ML-DSA o SLH-DSA, según restricciones.
4. Añadir rotación, versiones de protocolos y pruebas de interoperabilidad antes del despliegue total.

### Conclusión

La transición a la criptografía post-cuántica no consiste solo en cambiar una función: requiere un plan de criptoagilidad, pruebas y revisión del protocolo global.